# CIOS Document Extraction Service

External microservice for extracting clean text from PDFs, SEC filings, PubMed articles, and complex HTML sources.

**How it works:**
1. Run this notebook in Google Colab
2. It starts a Flask server with a localtunnel public URL
3. Copy the URL into CIOS System → Configuration → Document Extraction Service
4. CIOS sends document URLs here for full-text extraction
5. Returns clean text — CIOS then runs extraction prompts against real document text

**No account required.** localtunnel needs no signup, no auth token, no API key.

**Zero-fabrication guarantee:** LLM reads actual document text, sourceQuote is copied verbatim. If the data isn't in the document, LLM returns null. Null is correct — invention is not.

In [ ]:
# CELL 1 — Install dependencies
!pip install flask requests pdfminer.six beautifulsoup4 -q
!npm install -g localtunnel
print('All dependencies installed.')

In [ ]:
# CELL 2 — Build the Flask extraction service
from flask import Flask, request, jsonify
import requests as http_requests
from pdfminer.high_level import extract_text as pdf_extract_text
from bs4 import BeautifulSoup
import io
import time
import traceback

app = Flask(__name__)

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
MAX_TEXT_LENGTH = 50000

def clean_html(html_text):
    """Extract clean text from HTML, removing nav/footer/ads/scripts."""
    soup = BeautifulSoup(html_text, 'html.parser')
    for tag in soup(['nav', 'footer', 'header', 'script', 'style', 'iframe',
                     'noscript', 'svg', 'form']):
        tag.decompose()
    for selector in ['.cookie-banner', '.nav', '.footer', '.sidebar',
                     '.advertisement', '.ad', '#cookie-consent',
                     '.social-share', '.related-articles']:
        for el in soup.select(selector):
            el.decompose()
    article = soup.find('article') or soup.find(attrs={'role': 'main'}) or soup.find(class_='content')
    if article:
        text = article.get_text(separator=' ', strip=True)
    else:
        text = soup.get_text(separator=' ', strip=True)
    title_tag = soup.find('title')
    title = title_tag.get_text(strip=True) if title_tag else ''
    return text[:MAX_TEXT_LENGTH], title[:300]


def extract_pdf(content_bytes):
    """Extract text from PDF bytes using pdfminer."""
    text = pdf_extract_text(io.BytesIO(content_bytes))
    return text[:MAX_TEXT_LENGTH] if text else ''


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'service': 'cios-document-extractor', 'timestamp': time.time()})


@app.route('/extract', methods=['POST'])
def extract():
    """Fetch a URL and extract clean text. Handles PDF and HTML."""
    try:
        data = request.json or {}
        url = data.get('url', '')
        if not url:
            return jsonify({'error': 'Missing url parameter'}), 400

        start = time.time()
        resp = http_requests.get(url, headers={'User-Agent': USER_AGENT}, timeout=30, allow_redirects=True)
        resp.raise_for_status()

        content_type = resp.headers.get('content-type', '').lower()
        is_pdf = 'pdf' in content_type or url.lower().endswith('.pdf')

        if is_pdf:
            text = extract_pdf(resp.content)
            title = url.split('/')[-1].replace('.pdf', '') if '/' in url else 'PDF Document'
            doc_type = 'pdf'
        else:
            text, title = clean_html(resp.text)
            doc_type = 'html'

        elapsed = time.time() - start
        return jsonify({
            'url': resp.url,
            'text': text,
            'title': title,
            'length': len(text),
            'type': doc_type,
            'content_type': content_type,
            'elapsed_ms': int(elapsed * 1000),
            'status': 'ok'
        })
    except Exception as e:
        traceback.print_exc()
        return jsonify({'error': str(e), 'status': 'error'}), 500


@app.route('/pubmed', methods=['POST'])
def pubmed():
    """Fetch PubMed abstract via NCBI E-utilities API. Returns exactly what NCBI has — no hallucination."""
    try:
        data = request.json or {}
        pmid = data.get('pmid', '')
        if not pmid:
            return jsonify({'error': 'Missing pmid parameter'}), 400

        efetch_url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id={pmid}&rettype=abstract&retmode=text'
        resp = http_requests.get(efetch_url, timeout=15)
        resp.raise_for_status()
        abstract_text = resp.text

        esummary_url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&id={pmid}&retmode=json'
        meta_resp = http_requests.get(esummary_url, timeout=10)
        title = f'PMID {pmid}'
        journal = ''
        pub_date = ''
        if meta_resp.ok:
            try:
                meta = meta_resp.json()
                article = meta.get('result', {}).get(str(pmid), {})
                title = article.get('title', title)
                journal = article.get('fulljournalname', '')
                pub_date = article.get('pubdate', '')
            except:
                pass

        return jsonify({
            'pmid': pmid,
            'text': abstract_text,
            'title': title,
            'journal': journal,
            'pub_date': pub_date,
            'length': len(abstract_text),
            'source': 'pubmed_api',
            'status': 'ok'
        })
    except Exception as e:
        traceback.print_exc()
        return jsonify({'error': str(e), 'status': 'error'}), 500


@app.route('/batch', methods=['POST'])
def batch_extract():
    """Extract text from multiple URLs in one call."""
    try:
        data = request.json or {}
        urls = data.get('urls', [])
        if not urls:
            return jsonify({'error': 'Missing urls parameter'}), 400

        results = []
        for url in urls[:10]:
            try:
                resp = http_requests.get(url, headers={'User-Agent': USER_AGENT}, timeout=30, allow_redirects=True)
                resp.raise_for_status()
                content_type = resp.headers.get('content-type', '').lower()
                is_pdf = 'pdf' in content_type or url.lower().endswith('.pdf')
                if is_pdf:
                    text = extract_pdf(resp.content)
                    title = url.split('/')[-1].replace('.pdf', '')
                else:
                    text, title = clean_html(resp.text)
                results.append({'url': resp.url, 'text': text, 'title': title, 'length': len(text), 'status': 'ok'})
            except Exception as e:
                results.append({'url': url, 'text': '', 'title': '', 'length': 0, 'status': 'error', 'error': str(e)})

        return jsonify({'results': results, 'count': len(results)})
    except Exception as e:
        return jsonify({'error': str(e)}), 500


print('Flask app defined with endpoints: /health, /extract, /pubmed, /batch')

In [ ]:
# CELL 3 — Start Flask and expose via localtunnel
import subprocess
import threading
import time
import re

def run_flask():
    app.run(port=5000, use_reloader=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()
time.sleep(2)
print('Flask server started on port 5000')

lt_proc = subprocess.Popen(
    ['lt', '--port', '5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
for _ in range(30):
    line = lt_proc.stdout.readline()
    if line:
        match = re.search(r'(https?://\S+\.loca\.lt)', line)
        if match:
            public_url = match.group(1)
            break
    time.sleep(0.5)

if public_url:
    print(f'\n{"="*60}')
    print(f'CIOS Document Extraction Service is LIVE')
    print(f'{"="*60}')
    print(f'\nPublic URL: {public_url}')
    print(f'\nEndpoints:')
    print(f'  GET  {public_url}/health     — Health check')
    print(f'  POST {public_url}/extract     — Extract text from any URL (PDF/HTML)')
    print(f'  POST {public_url}/pubmed      — Fetch PubMed abstract by PMID')
    print(f'  POST {public_url}/batch       — Batch extract from multiple URLs')
    print(f'\n>>> Copy this URL into CIOS System → Configuration → Document Extraction Service')
    print(f'>>> URL: {public_url}')
    print(f'{"="*60}\n')
else:
    print('ERROR: Could not get localtunnel URL. Check output above for errors.')
    if lt_proc.poll() is not None:
        print(f'localtunnel exited with code {lt_proc.returncode}')
        print(lt_proc.stderr.read())

In [ ]:
# CELL 4 — Test: Local extraction
import requests as test_requests

test = test_requests.post(
    'http://localhost:5000/extract',
    json={'url': 'https://ir.insmed.com/news-releases'}
)
result = test.json()
print(f'Status: {test.status_code}')
print(f'Type: {result.get("type")}')
print(f'Text length: {result.get("length")} chars')
print(f'Elapsed: {result.get("elapsed_ms")}ms')
print(f'\nFirst 300 chars:')
print(result.get('text', '')[:300])

In [ ]:
# CELL 5 — Test: PubMed CONVERT trial (PMID 29906251)
test_resp = test_requests.post('http://localhost:5000/pubmed', json={'pmid': '29906251'})
result = test_resp.json()
print(f'PubMed Test — PMID 29906251')
print(f'Status: {result.get("status")}')
print(f'Title: {result.get("title")}')
print(f'Journal: {result.get("journal")}')
print(f'Text length: {result.get("length")} chars')
print(f'\nFirst 500 chars:')
print(result.get('text', '')[:500])